# Short-Volshock Metric-Aligned Checkpoint/BC Teacher v1

This notebook continues from `official_replay_metric_aligned_checkpoint_bc_screen_v1`. The previous screen found no promotable official replay candidate and selected `short_volshock_cap05_hit2` only as the lowest-loss near-miss reference.

Governance:
- Baselines: canonical SOXLBacktest qlib reference `soxl_backtest_policy_reimplementation_v1`; direct comparison `official_replay_metric_aligned_checkpoint_bc_screen_v1`.
- Role: `diagnose` / `checkpoint_teacher_contract`; no actor promotion or PPO continuation is attempted here.
- Target windows: official warm-start `wf_2022`, `wf_2023_2024`; final retention still requires `wf_2020`, `wf_2022`, `wf_2023_2024`, strict return alignment, and MaxDD below 20%.
- Changed layer: model-selection objective / target-generation handoff only.
- Hypothesis: the short-volshock near-miss is useful as a metric-aligned checkpoint/BC teacher, but it must be converted into fold-local target or reward/checkpoint terms rather than copied as eval-window training labels.
- Acceptance: produce a non-leaking teacher contract with explicit gaps, trace blockers, source daily paths, and next gates. `promotion_allowed` must remain false.


In [1]:
from __future__ import annotations

import json
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display


def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for cur in (start, *start.parents):
        if (cur / "AGENTS.md").exists() and (cur / "SPEC.md").exists():
            return cur
    raise FileNotFoundError("Could not find repo root containing AGENTS.md and SPEC.md")


ROOT = find_repo_root(Path.cwd())
CARD = ROOT / "runtime/model_selection/card/short_volshock_metric_aligned_checkpoint_or_bc_teacher_v1"
CARD.mkdir(parents=True, exist_ok=True)

SOURCE_CARD = ROOT / "runtime/model_selection/card/short_volshock_official_replay_screen_v1"
SCREEN_CARD = ROOT / "runtime/model_selection/card/official_replay_metric_aligned_checkpoint_bc_screen_v1"
TARGET_RELATIVE_RETURN_PP = 30.0
TARGET_MAX_DD_PCT = 15.0
FINAL_MAX_DD_PCT = 20.0
FINAL_REQUIRED_WINDOWS = {"wf_2020", "wf_2022", "wf_2023_2024"}
SELECTED_VARIANT = "short_volshock_cap05_hit2"
SELECTED_CANDIDATE = "multi_obj_floor45_cap20_riskfirst"
EXPERIMENT = "short_volshock_metric_aligned_checkpoint_or_bc_teacher_v1"


def rel(path: Path) -> str:
    return str(path.relative_to(ROOT))


def load_json(path: Path):
    if not path.exists():
        return None
    return json.loads(path.read_text())


def summary_of(obj):
    if isinstance(obj, dict) and isinstance(obj.get("summary"), dict):
        return obj["summary"]
    return obj if isinstance(obj, dict) else {}


def write_json(name: str, payload) -> str:
    path = CARD / name
    path.write_text(json.dumps(payload, indent=2, sort_keys=True, ensure_ascii=True) + "\n")
    return rel(path)


def write_csv(name: str, frame: pd.DataFrame) -> str:
    path = CARD / name
    frame.to_csv(path, index=False)
    return rel(path)

source_files = {
    "screen_selection": SCREEN_CARD / "selection.json",
    "screen_summary": SCREEN_CARD / "summary.json",
    "screen_scored_pool": SCREEN_CARD / "scored_official_candidate_pool.csv",
    "screen_selected_fold_metrics": SCREEN_CARD / "selected_candidate_fold_metrics.csv",
    "source_selection": SOURCE_CARD / "selection.json",
    "source_summary": SOURCE_CARD / "summary.json",
    "source_candidate_summary": SOURCE_CARD / "official_replay_candidate_summary.csv",
    "source_selected_fold_metrics": SOURCE_CARD / "selected_official_replay_fold_metrics.csv",
    "source_selected_trace": SOURCE_CARD / "selected_decision_trace_summary.csv",
    "source_all_metrics": SOURCE_CARD / "official_replay_metrics.csv",
    "source_all_traces": SOURCE_CARD / "decision_trace_summary.csv",
    "strict_retention_gate": ROOT / "runtime/model_selection/card/model_retention_walkforward_gate_v1/summary.json",
    "final_retention_contract": ROOT / "runtime/model_selection/card/final_walkforward_retention_contract_v1/summary.json",
}
source_manifest = [
    {"name": name, "path": rel(path), "exists": path.exists(), "bytes": path.stat().st_size if path.exists() else 0}
    for name, path in source_files.items()
]
source_manifest_path = write_json("source_manifest.json", source_manifest)
print(f"repo_root={ROOT}")
print(f"source_manifest={source_manifest_path}")
display(pd.DataFrame(source_manifest))

repo_root=/Users/rziiiii/dev/workspace
source_manifest=runtime/model_selection/card/short_volshock_metric_aligned_checkpoint_or_bc_teacher_v1/source_manifest.json


,name,path,exists,bytes
0,screen_selection,runtime/model_selection/card/official_replay_m...,True,3645
1,screen_summary,runtime/model_selection/card/official_replay_m...,True,4784
2,screen_scored_pool,runtime/model_selection/card/official_replay_m...,True,9714
3,screen_selected_fold_metrics,runtime/model_selection/card/official_replay_m...,True,681
4,source_selection,runtime/model_selection/card/short_volshock_of...,True,7112
5,source_summary,runtime/model_selection/card/short_volshock_of...,True,9066
6,source_candidate_summary,runtime/model_selection/card/short_volshock_of...,True,1787
7,source_selected_fold_metrics,runtime/model_selection/card/short_volshock_of...,True,2048
8,source_selected_trace,runtime/model_selection/card/short_volshock_of...,True,747
9,source_all_metrics,runtime/model_selection/card/short_volshock_of...,True,6320


## Selected Near-Miss Metrics

The candidate is a teacher reference only. It passes `wf_2022`, misses `wf_2023_2024` by a small amount, and lacks `wf_2020` coverage.

In [2]:
screen_selection = load_json(source_files["screen_selection"]) or {}
screen_summary = load_json(source_files["screen_summary"]) or {}
retention_summary = summary_of(load_json(source_files["strict_retention_gate"]) or {})
final_summary = summary_of(load_json(source_files["final_retention_contract"]) or {})
selected_fold_metrics = pd.read_csv(source_files["source_selected_fold_metrics"])
selected_trace = pd.read_csv(source_files["source_selected_trace"])
candidate_summary = pd.read_csv(source_files["source_candidate_summary"])
all_metrics = pd.read_csv(source_files["source_all_metrics"])
all_traces = pd.read_csv(source_files["source_all_traces"])

selected_fold_metrics = selected_fold_metrics[
    (selected_fold_metrics["variant"] == SELECTED_VARIANT)
    & (selected_fold_metrics["candidate"] == SELECTED_CANDIDATE)
].copy()
if selected_fold_metrics.empty:
    raise RuntimeError("selected near-miss fold metrics not found")

selected_fold_metrics["relative_gap_pp"] = (TARGET_RELATIVE_RETURN_PP - selected_fold_metrics["official_warm_relative_return"]).clip(lower=0.0)
selected_fold_metrics["max_dd_gap_pct"] = (selected_fold_metrics["official_warm_max_dd"] - TARGET_MAX_DD_PCT).clip(lower=0.0)
selected_fold_metrics["final_20pct_max_dd_gap_pct"] = (selected_fold_metrics["official_warm_max_dd"] - FINAL_MAX_DD_PCT).clip(lower=0.0)
selected_fold_metrics["teacher_status"] = selected_fold_metrics.apply(
    lambda row: "pass" if bool(row["full_gate"]) else ("near_miss" if row["relative_gap_pp"] <= 1.0 and row["max_dd_gap_pct"] <= 0.5 else "fail"),
    axis=1,
)
selected_fold_metrics["teacher_usage"] = "official_replay_reference_only_not_training_label"
selected_fold_metrics_path = write_csv("selected_teacher_fold_metrics.csv", selected_fold_metrics)

present_windows = set(selected_fold_metrics["fold"].astype(str))
coverage = pd.DataFrame([
    {
        "candidate": SELECTED_VARIANT,
        "present_windows": ";".join(sorted(present_windows)),
        "missing_final_required_windows": ";".join(sorted(FINAL_REQUIRED_WINDOWS - present_windows)),
        "final_window_coverage_gate": FINAL_REQUIRED_WINDOWS.issubset(present_windows),
        "official_two_window_gate_pass_count": int(selected_fold_metrics["full_gate"].astype(bool).sum()),
        "official_two_window_count": int(len(selected_fold_metrics)),
    }
])
coverage_path = write_csv("teacher_window_coverage.csv", coverage)
print(f"selected_teacher_fold_metrics={selected_fold_metrics_path}")
print(f"teacher_window_coverage={coverage_path}")
display(selected_fold_metrics[[
    "fold", "official_warm_relative_return", "official_warm_max_dd", "official_warm_avg_pos", "trades", "full_gate",
    "relative_gap_pp", "max_dd_gap_pct", "final_20pct_max_dd_gap_pct", "teacher_status", "peak_date", "trough_date", "daily_path"
]])
display(coverage)

selected_teacher_fold_metrics=runtime/model_selection/card/short_volshock_metric_aligned_checkpoint_or_bc_teacher_v1/selected_teacher_fold_metrics.csv
teacher_window_coverage=runtime/model_selection/card/short_volshock_metric_aligned_checkpoint_or_bc_teacher_v1/teacher_window_coverage.csv


,fold,official_warm_relative_return,official_warm_max_dd,official_warm_avg_pos,trades,full_gate,relative_gap_pp,max_dd_gap_pct,final_20pct_max_dd_gap_pct,teacher_status,peak_date,trough_date,daily_path
0,wf_2022,83.564183,8.746809,0.062850,29,True,0.000000,0.000000,0.0,pass,2022-01-03,2022-08-09,runtime/model_selection/card/short_volshock_of...
1,wf_2023_2024,29.817263,15.107381,0.207768,49,False,0.182737,0.107381,0.0,near_miss,2024-03-07,2024-05-01,runtime/model_selection/card/short_volshock_of...


,candidate,present_windows,missing_final_required_windows,final_window_coverage_gate,official_two_window_gate_pass_count,official_two_window_count
0,short_volshock_cap05_hit2,wf_2022;wf_2023_2024,wf_2020,False,1,2


## Trace Absorption

The selected action cap changes many target positions, but only a small share becomes orders. This is why the handoff goes to model/checkpoint target design, not another runtime threshold tweak.

In [3]:
selected_trace = selected_trace[
    (selected_trace["variant"] == SELECTED_VARIANT)
    & (selected_trace["candidate"] == SELECTED_CANDIDATE)
].copy()
if selected_trace.empty:
    raise RuntimeError("selected decision trace summary not found")
for col in ["overlay_gate_ready_days", "cap_changed_days", "cap_order_days", "blocked_by_trade_lock_days", "blocked_by_min_rebalance_days", "sell_order_days", "trace_rows"]:
    selected_trace[col] = pd.to_numeric(selected_trace[col], errors="coerce")
selected_trace["cap_order_share_of_changed"] = selected_trace["cap_order_days"] / selected_trace["cap_changed_days"].replace(0, pd.NA)
selected_trace["cap_changed_share_of_ready"] = selected_trace["cap_changed_days"] / selected_trace["overlay_gate_ready_days"].replace(0, pd.NA)
selected_trace["blocked_by_min_rebalance_share"] = selected_trace["blocked_by_min_rebalance_days"] / selected_trace["trace_rows"].replace(0, pd.NA)
selected_trace["blocked_by_trade_lock_share"] = selected_trace["blocked_by_trade_lock_days"] / selected_trace["trace_rows"].replace(0, pd.NA)
selected_trace["trace_interpretation"] = selected_trace.apply(
    lambda row: "low_order_absorption_use_state_checkpoint_not_more_overlay" if row["cap_order_share_of_changed"] < 0.20 else "order_absorption_available",
    axis=1,
)
trace_path = write_csv("teacher_trace_absorption.csv", selected_trace)
print(f"teacher_trace_absorption={trace_path}")
display(selected_trace[[
    "fold", "trace_rows", "overlay_gate_ready_days", "cap_changed_days", "cap_order_days",
    "cap_order_share_of_changed", "blocked_by_min_rebalance_share", "blocked_by_trade_lock_share", "sell_order_days", "trace_interpretation", "trace_path"
]])

teacher_trace_absorption=runtime/model_selection/card/short_volshock_metric_aligned_checkpoint_or_bc_teacher_v1/teacher_trace_absorption.csv


,fold,trace_rows,overlay_gate_ready_days,cap_changed_days,cap_order_days,cap_order_share_of_changed,blocked_by_min_rebalance_share,blocked_by_trade_lock_share,sell_order_days,trace_interpretation,trace_path
0,wf_2022,308,236,236,17,0.072034,0.905844,0.162338,13,low_order_absorption_use_state_checkpoint_not_...,runtime/model_selection/card/short_volshock_of...
1,wf_2023_2024,309,64,58,8,0.137931,0.841424,0.365696,25,low_order_absorption_use_state_checkpoint_not_...,runtime/model_selection/card/short_volshock_of...


## Failing Interval Teacher Path

For the failing `wf_2023_2024` fold, compare selected near-miss positions against the source baseline inside the worst drawdown interval. This is diagnostic input for checkpoint/target design only.

In [4]:
def read_daily(rel_path: str, prefix: str) -> pd.DataFrame:
    frame = pd.read_csv(ROOT / rel_path, parse_dates=["datetime"])
    keep = ["datetime", "strategy_nav", "price", "buyhold_nav", "position", "excess_pct", "action"]
    frame = frame[[col for col in keep if col in frame.columns]].copy()
    return frame.rename(columns={col: f"{prefix}_{col}" for col in frame.columns if col != "datetime"})

failing_row = selected_fold_metrics[selected_fold_metrics["teacher_status"] != "pass"].sort_values(["relative_gap_pp", "max_dd_gap_pct"], ascending=False).iloc[0]
failing_fold = str(failing_row["fold"])
selected_daily_path = str(failing_row["daily_path"])
source_row = all_metrics[
    (all_metrics["candidate"] == SELECTED_CANDIDATE)
    & (all_metrics["fold"] == failing_fold)
    & (all_metrics["variant"] == "baseline_no_overlay")
].iloc[0]
source_daily_path = str(source_row["daily_path"])
selected_daily = read_daily(selected_daily_path, "selected")
source_daily = read_daily(source_daily_path, "source")
comparison = source_daily.merge(selected_daily, on="datetime", how="inner")
comparison["position_delta_selected_minus_source"] = comparison["selected_position"] - comparison["source_position"]
comparison["nav_delta_selected_minus_source"] = comparison["selected_strategy_nav"] - comparison["source_strategy_nav"]
comparison["excess_pct_delta_selected_minus_source"] = comparison["selected_excess_pct"] - comparison["source_excess_pct"]
peak = pd.to_datetime(failing_row["peak_date"])
trough = pd.to_datetime(failing_row["trough_date"])
interval = comparison[(comparison["datetime"] >= peak) & (comparison["datetime"] <= trough)].copy()
interval_summary = pd.DataFrame([
    {
        "fold": failing_fold,
        "peak_date": str(failing_row["peak_date"]),
        "trough_date": str(failing_row["trough_date"]),
        "interval_rows": int(len(interval)),
        "source_avg_position": float(interval["source_position"].mean()),
        "selected_avg_position": float(interval["selected_position"].mean()),
        "avg_position_delta": float(interval["position_delta_selected_minus_source"].mean()),
        "min_position_delta": float(interval["position_delta_selected_minus_source"].min()),
        "max_position_delta": float(interval["position_delta_selected_minus_source"].max()),
        "selected_relative_gap_pp": float(failing_row["relative_gap_pp"]),
        "selected_max_dd_gap_pct": float(failing_row["max_dd_gap_pct"]),
        "selected_daily_path": selected_daily_path,
        "source_daily_path": source_daily_path,
    }
])
comparison_path = write_csv("selected_vs_source_daily_path.csv", comparison)
interval_path = write_csv("failing_interval_selected_vs_source.csv", interval)
interval_summary_path = write_csv("failing_interval_summary.csv", interval_summary)
print(f"selected_vs_source_daily_path={comparison_path}")
print(f"failing_interval_selected_vs_source={interval_path}")
print(f"failing_interval_summary={interval_summary_path}")
display(interval_summary)
display(interval.head(12))

selected_vs_source_daily_path=runtime/model_selection/card/short_volshock_metric_aligned_checkpoint_or_bc_teacher_v1/selected_vs_source_daily_path.csv
failing_interval_selected_vs_source=runtime/model_selection/card/short_volshock_metric_aligned_checkpoint_or_bc_teacher_v1/failing_interval_selected_vs_source.csv
failing_interval_summary=runtime/model_selection/card/short_volshock_metric_aligned_checkpoint_or_bc_teacher_v1/failing_interval_summary.csv


,fold,peak_date,trough_date,interval_rows,source_avg_position,selected_avg_position,avg_position_delta,min_position_delta,max_position_delta,selected_relative_gap_pp,selected_max_dd_gap_pct,selected_daily_path,source_daily_path
0,wf_2023_2024,2024-03-07,2024-05-01,39,0.210101,0.137229,-0.072873,-0.351264,0.005279,0.182737,0.107381,runtime/model_selection/card/short_volshock_of...,runtime/model_selection/card/short_volshock_of...


,datetime,source_strategy_nav,source_price,source_buyhold_nav,source_position,source_excess_pct,source_action,selected_strategy_nav,selected_price,selected_buyhold_nav,selected_position,selected_excess_pct,selected_action,position_delta_selected_minus_source,nav_delta_selected_minus_source,excess_pct_delta_selected_minus_source
45,2024-03-07,1.434078e+06,55.320000,1.972896e+06,0.512402,-27.311003,HOLD,1.434078e+06,55.320000,1.972896e+06,0.512402,-27.311003,HOLD,0.000000,0.000000,0.000000
46,2024-03-08,1.331341e+06,48.950001,1.745720e+06,0.187500,-23.736862,SELL,1.331341e+06,48.950001,1.745720e+06,0.187500,-23.736862,SELL,0.000000,0.000000,0.000000
47,2024-03-11,1.320126e+06,46.770000,1.667974e+06,0.182256,-20.854508,HOLD,1.320126e+06,46.770000,1.667974e+06,0.182256,-20.854508,HOLD,0.000000,0.000000,0.000000
48,2024-03-12,1.334582e+06,49.580002,1.768188e+06,0.191114,-24.522628,HOLD,1.334582e+06,49.580002,1.768188e+06,0.191114,-24.522628,HOLD,0.000000,0.000000,0.000000
49,2024-03-13,1.316577e+06,46.080002,1.643367e+06,0.180051,-19.885385,HOLD,1.316577e+06,46.080002,1.643367e+06,0.180051,-19.885385,HOLD,0.000000,0.000000,0.000000
50,2024-03-14,1.303459e+06,43.529999,1.552425e+06,0.171799,-16.037249,HOLD,1.303459e+06,43.529999,1.552425e+06,0.171799,-16.037249,HOLD,0.000000,0.000000,0.000000
51,2024-03-15,1.299395e+06,42.740002,1.524251e+06,0.169209,-14.751923,HOLD,1.299395e+06,42.740002,1.524251e+06,0.169209,-14.751923,HOLD,0.000000,0.000000,0.000000
52,2024-03-18,1.295869e+06,42.540001,1.517118e+06,0.100000,-14.583532,SELL,1.294085e+06,42.540001,1.517118e+06,0.050000,-14.701091,SELL,-0.050000,-1783.511165,-0.117559
53,2024-03-19,1.292515e+06,41.419998,1.477175e+06,0.095961,-12.500900,HOLD,1.292452e+06,41.419998,1.477175e+06,0.046753,-12.505206,HOLD,-0.049208,-63.605633,-0.004306
54,2024-03-20,1.298444e+06,43.400002,1.547789e+06,0.100089,-16.109735,HOLD,1.295340e+06,43.400002,1.547789e+06,0.048878,-16.310289,HOLD,-0.051211,-3104.150610,-0.200554


## Teacher Recipe

The recipe is a contract for the next checkpoint/reward/BC-target screen. It explicitly forbids copying eval-window positions into training labels.

In [5]:
selected_candidate_summary = candidate_summary[candidate_summary["variant"] == SELECTED_VARIANT].iloc[0].to_dict()
max_relative_gap = float(selected_fold_metrics["relative_gap_pp"].max())
max_dd_gap = float(selected_fold_metrics["max_dd_gap_pct"].max())
teacher_recipe = {
    "experiment": EXPERIMENT,
    "created_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "source_candidate": SELECTED_CANDIDATE,
    "source_variant": SELECTED_VARIANT,
    "source_screen": "short_volshock_official_replay_screen_v1",
    "source_selection_screen": "official_replay_metric_aligned_checkpoint_bc_screen_v1",
    "teacher_usage": "checkpoint_scoring_or_fold_local_target_generation_reference_only",
    "do_not_use_as": [
        "promoted_actor",
        "runtime_overlay_promotion",
        "eval_window_training_label_copy",
        "permission_for_longer_ppo",
    ],
    "target_contract": {
        "official_eval_relative_return_pp": TARGET_RELATIVE_RETURN_PP,
        "official_eval_max_dd_pct": TARGET_MAX_DD_PCT,
        "final_retention_max_dd_pct": FINAL_MAX_DD_PCT,
        "final_required_windows": sorted(FINAL_REQUIRED_WINDOWS),
        "missing_final_required_windows": sorted(FINAL_REQUIRED_WINDOWS - present_windows),
        "required_best_return_pct": retention_summary.get("required_best_return_pct"),
        "current_best_return_pct": retention_summary.get("current_best_return_pct"),
        "reference_profile": retention_summary.get("reference_profile"),
    },
    "selected_gap": {
        "max_relative_gap_pp": max_relative_gap,
        "max_dd_gap_pct": max_dd_gap,
        "near_miss_total_gate_loss": float(screen_summary.get("selected", {}).get("selected_total_gate_loss") or selected_candidate_summary.get("gate_loss")),
        "failing_fold": failing_fold,
        "failing_fold_relative_return_pp": float(failing_row["official_warm_relative_return"]),
        "failing_fold_max_dd_pct": float(failing_row["official_warm_max_dd"]),
    },
    "trace_absorption": selected_trace[[
        "fold", "overlay_gate_ready_days", "cap_changed_days", "cap_order_days",
        "cap_order_share_of_changed", "blocked_by_min_rebalance_share", "blocked_by_trade_lock_share",
        "trace_interpretation",
    ]].to_dict(orient="records"),
    "objective_terms_for_next_screen": [
        "official_replay_relative_gap_pp",
        "official_replay_max_dd_gap_pct",
        "short_volshock_state_deleverage_consistency",
        "trend_preservation_floor_for_failed_underparticipation_windows",
        "checkpoint_objective_loss_prioritizing_official_daily_path_metrics",
        "wf2020_coverage_gate_before_actor_retention",
    ],
    "leakage_guard": {
        "eval_window_positions_are_diagnostic_only": True,
        "next_target_generation_must_be_fold_local": True,
        "eval_overlap_rows_required": 0,
        "copy_selected_daily_positions_into_training_labels": False,
    },
    "next_mechanism": "short_volshock_teacher_reward_checkpoint_screen_v1",
}
teacher_recipe_path = write_json("teacher_checkpoint_bc_recipe.json", teacher_recipe)
print(f"teacher_checkpoint_bc_recipe={teacher_recipe_path}")
print(json.dumps(teacher_recipe, indent=2, sort_keys=True))

teacher_checkpoint_bc_recipe=runtime/model_selection/card/short_volshock_metric_aligned_checkpoint_or_bc_teacher_v1/teacher_checkpoint_bc_recipe.json
{
  "created_at": "2026-06-16T03:22:22+00:00",
  "do_not_use_as": [
    "promoted_actor",
    "runtime_overlay_promotion",
    "eval_window_training_label_copy",
    "permission_for_longer_ppo"
  ],
  "experiment": "short_volshock_metric_aligned_checkpoint_or_bc_teacher_v1",
  "leakage_guard": {
    "copy_selected_daily_positions_into_training_labels": false,
    "eval_overlap_rows_required": 0,
    "eval_window_positions_are_diagnostic_only": true,
    "next_target_generation_must_be_fold_local": true
  },
  "next_mechanism": "short_volshock_teacher_reward_checkpoint_screen_v1",
  "objective_terms_for_next_screen": [
    "official_replay_relative_gap_pp",
    "official_replay_max_dd_gap_pct",
    "short_volshock_state_deleverage_consistency",
    "trend_preservation_floor_for_failed_underparticipation_windows",
    "checkpoint_objective_

## Summary And Decision

The selected near-miss is useful enough to preserve as a teacher/checkpoint contract, but all promotion and actor retention gates remain closed.

In [6]:
decision = "short_volshock_metric_aligned_teacher_ready_no_promotion"
summary = {
    "experiment": EXPERIMENT,
    "created_at": teacher_recipe["created_at"],
    "decision": decision,
    "promotion_allowed": False,
    "actor_retention_allowed": False,
    "ppo_allowed_next": False,
    "checkpoint_teacher_allowed_next": True,
    "selected_candidate": SELECTED_VARIANT,
    "selected_source_candidate": SELECTED_CANDIDATE,
    "selected_fold_count": int(len(selected_fold_metrics)),
    "selected_full_gate_count": int(selected_fold_metrics["full_gate"].astype(bool).sum()),
    "selected_teacher_status_counts": selected_fold_metrics["teacher_status"].value_counts().to_dict(),
    "selected_max_relative_gap_pp": max_relative_gap,
    "selected_max_dd_gap_pct": max_dd_gap,
    "missing_final_required_windows": sorted(FINAL_REQUIRED_WINDOWS - present_windows),
    "strict_retention_context": {
        "final_retained_count": final_summary.get("final_retained_count"),
        "strict_passing_profile_count": final_summary.get("strict_passing_profile_count") or retention_summary.get("passing_profile_count"),
        "required_best_return_pct": retention_summary.get("required_best_return_pct"),
        "current_best_return_pct": retention_summary.get("current_best_return_pct"),
        "max_walkforward_drawdown_pct": retention_summary.get("max_walkforward_drawdown_pct"),
    },
    "trace_decision": "use_state_checkpoint_or_target_generation_not_more_runtime_overlay_thresholds",
    "next_mechanism": teacher_recipe["next_mechanism"],
    "sidecars": {
        "source_manifest": source_manifest_path,
        "selected_teacher_fold_metrics": selected_fold_metrics_path,
        "teacher_window_coverage": coverage_path,
        "teacher_trace_absorption": trace_path,
        "selected_vs_source_daily_path": comparison_path,
        "failing_interval_selected_vs_source": interval_path,
        "failing_interval_summary": interval_summary_path,
        "teacher_checkpoint_bc_recipe": teacher_recipe_path,
        "summary": f"runtime/model_selection/card/{EXPERIMENT}/summary.json",
        "selection": f"runtime/model_selection/card/{EXPERIMENT}/selection.json",
    },
}
selection = {
    **summary,
    "selected_daily_paths": selected_fold_metrics[["fold", "daily_path"]].to_dict(orient="records"),
    "source_baseline_failing_daily_path": source_daily_path,
    "selected_failing_daily_path": selected_daily_path,
    "hard_blockers": [
        "wf_2020 coverage missing",
        "official two-window full gate not passed",
        "strict final retained actor count remains zero",
        "eval-window path cannot be copied into training labels",
    ],
}
summary_path = write_json("summary.json", summary)
selection_path = write_json("selection.json", selection)
print(f"summary={summary_path}")
print(f"selection={selection_path}")
print(json.dumps(summary, indent=2, sort_keys=True))

summary=runtime/model_selection/card/short_volshock_metric_aligned_checkpoint_or_bc_teacher_v1/summary.json
selection=runtime/model_selection/card/short_volshock_metric_aligned_checkpoint_or_bc_teacher_v1/selection.json
{
  "actor_retention_allowed": false,
  "checkpoint_teacher_allowed_next": true,
  "created_at": "2026-06-16T03:22:22+00:00",
  "decision": "short_volshock_metric_aligned_teacher_ready_no_promotion",
  "experiment": "short_volshock_metric_aligned_checkpoint_or_bc_teacher_v1",
  "missing_final_required_windows": [
    "wf_2020"
  ],
  "next_mechanism": "short_volshock_teacher_reward_checkpoint_screen_v1",
  "ppo_allowed_next": false,
  "promotion_allowed": false,
  "selected_candidate": "short_volshock_cap05_hit2",
  "selected_fold_count": 2,
  "selected_full_gate_count": 1,
  "selected_max_dd_gap_pct": 0.10738078525339922,
  "selected_max_relative_gap_pp": 0.18273669306148932,
  "selected_source_candidate": "multi_obj_floor45_cap20_riskfirst",
  "selected_teacher_status

In [7]:
conclusion_md = f"""
## Conclusion

Decision: `{decision}`.

`{SELECTED_VARIANT}` is preserved only as a metric-aligned teacher/checkpoint reference. It passes `wf_2022` but remains a near-miss in `{failing_fold}` with relative gap `{max_relative_gap:.3f}pp` and MaxDD gap `{max_dd_gap:.3f}%` against the `+30pp / <=15%` official replay target. It also lacks final `wf_2020` coverage, so actor retention remains blocked.

The trace evidence supports moving this signal into checkpoint/target design rather than adding another runtime overlay threshold: selected cap order absorption is low, with cap orders much smaller than cap-changed days and high min-rebalance blocking.

Next action: run `{teacher_recipe['next_mechanism']}` using this recipe as a checkpoint/reward/BC-target contract. The next screen must generate fold-local targets or checkpoint terms, add `wf_2020`, and keep strict retention gates closed until MaxDD `<20%` and current-best return alignment are proven.
""".strip()
(CARD / "conclusion.md").write_text(conclusion_md + "\n")
print(rel(CARD / "conclusion.md"))
display(Markdown(conclusion_md))

runtime/model_selection/card/short_volshock_metric_aligned_checkpoint_or_bc_teacher_v1/conclusion.md


## Conclusion

Decision: `short_volshock_metric_aligned_teacher_ready_no_promotion`.

`short_volshock_cap05_hit2` is preserved only as a metric-aligned teacher/checkpoint reference. It passes `wf_2022` but remains a near-miss in `wf_2023_2024` with relative gap `0.183pp` and MaxDD gap `0.107%` against the `+30pp / <=15%` official replay target. It also lacks final `wf_2020` coverage, so actor retention remains blocked.

The trace evidence supports moving this signal into checkpoint/target design rather than adding another runtime overlay threshold: selected cap order absorption is low, with cap orders much smaller than cap-changed days and high min-rebalance blocking.

Next action: run `short_volshock_teacher_reward_checkpoint_screen_v1` using this recipe as a checkpoint/reward/BC-target contract. The next screen must generate fold-local targets or checkpoint terms, add `wf_2020`, and keep strict retention gates closed until MaxDD `<20%` and current-best return alignment are proven.